# Assignment — Holistic Prompt Optimization

**Program:** Brilian Sistem Informasi Bootcamp  
**Session 25:** Prompt Optimization

## 🎯 Learning Objectives

By completing this assignment, you will:
1. Apply **Chain-of-Thought**, **Self-Check**, and **Retrieval-Aware** prompting on real automotive scenarios
2. Combine reasoning techniques with **engine tuning** (Temperature, Top-P, Top-K) and **cost optimization** (max_tokens, prompt compression)
3. Analyze trade-offs between Safe / Balanced / Creative parameter configurations

## 📅 Submission

| Item | Detail |
|---|---|
| Platform | Google Classroom |
| File naming | `NamaLengkap_Sesi25_Assignment.ipynb` |
| Deadline | _[insert deadline]_ |
| Late policy | _[insert policy]_ |

## ⚙️ Setup

> ⚠️ **IMPORTANT — Never hardcode your API key.** Always use Google Colab Secrets.

**How to set it up:**
1. Click the 🔑 icon on the left sidebar in Colab
2. Add a secret named `GOOGLE_API_KEY` with your Gemini API key from [Google AI Studio](https://aistudio.google.com/)
3. Toggle **Notebook access** ON


In [ ]:
!pip install -q google-generativeai


In [ ]:
import google.generativeai as genai
from google.colab import userdata
import textwrap

# 🔐 Always retrieve secrets from Colab Secrets — never hardcode
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API configured successfully.")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Please ensure GOOGLE_API_KEY is set in Colab Secrets.")


## 🛠️ Helper Function

This helper supports all engine parameters PLUS reports token usage so you can analyze cost efficiency.


In [ ]:
def generate(prompt, temp=0.7, top_p=0.9, top_k=40, max_tokens=512, label=""):
    """Generate text and report token usage for cost analysis."""
    model = genai.GenerativeModel('gemini-2.5-flash')

    config = genai.types.GenerationConfig(
        temperature=temp,
        top_p=top_p,
        top_k=top_k,
        max_output_tokens=max_tokens,
    )

    response = model.generate_content(prompt, generation_config=config)

    print(f"\n{'=' * 70}")
    print(f"  {label}")
    print(f"  Params: temp={temp}, top_p={top_p}, top_k={top_k}, max_tokens={max_tokens}")
    print(f"{'=' * 70}")
    print(textwrap.fill(response.text, width=80))

    if hasattr(response, 'usage_metadata') and response.usage_metadata:
        m = response.usage_metadata
        print(
            f"\n📊 Tokens — input: {m.prompt_token_count}, "
            f"output: {m.candidates_token_count}, total: {m.total_token_count}"
        )
    return response.text


---

# Section 1 — Chain-of-Thought (CoT)

**Skenario:** Tentukan harga trade-in untuk **Mitsubishi Xpander Ultimate 2019 A/T**, KM 65.000, pajak hidup sampai Agustus 2026, ada lecet di bumper depan, service history lengkap di bengkel resmi. Harga pasaran sejenis Rp 195–215jt.

Bandingkan output **tanpa CoT** vs **dengan CoT**.


In [ ]:
# --- WITHOUT CoT ---
basic_prompt = """
Tentukan harga trade-in untuk Mitsubishi Xpander Ultimate 2019 A/T,
KM 65.000, pajak hidup sampai Agustus 2026, ada lecet di bumper depan,
service history lengkap di bengkel resmi. Harga pasaran sejenis Rp 195-215jt.
Berapa rekomendasi harga trade-in?
"""

generate(basic_prompt, temp=0.3, label="Section 1 — WITHOUT CoT")


In [ ]:
# --- WITH CoT ---
cot_prompt = """
Tentukan harga trade-in untuk Mitsubishi Xpander Ultimate 2019 A/T,
KM 65.000, pajak hidup sampai Agustus 2026, ada lecet di bumper depan,
service history lengkap di bengkel resmi. Harga pasaran sejenis Rp 195-215jt.

Lakukan analisis bertahap berikut sebelum memberikan rekomendasi:
1. Identifikasi faktor-faktor yang memengaruhi harga (tahun, KM, kondisi fisik, service history, status pajak)
2. Bandingkan dengan rentang harga pasaran
3. Hitung depresiasi & adjustment kondisi (potongan untuk lecet bumper, premium untuk service history lengkap)
4. Berikan rekomendasi harga trade-in dengan justifikasi setiap angka

Format jawaban: Langkah 1, Langkah 2, Langkah 3, Langkah 4, lalu Rekomendasi Akhir.
"""

generate(cot_prompt, temp=0.3, label="Section 1 — WITH CoT")


### 🔍 Reflection — Section 1

_(Tulis jawaban singkat di sel markdown ini setelah menjalankan kedua versi)_

1. Faktor apa yang muncul di output CoT yang **tidak** muncul di output basic?
2. Output mana yang lebih bisa dipertanggungjawabkan ke customer?
3. Berapa selisih total token antara kedua versi? Apakah CoT worth it untuk use case ini?


---

# Section 2 — Self-Check

**Skenario:** Customer test drive **Mitsubishi XForce Ultimate** kemarin tapi belum membuat keputusan pembelian. Buat email follow-up dan minta AI mengkritik draft-nya sendiri.


In [ ]:
# --- INITIAL DRAFT ---
initial_prompt = """
Tulis email follow-up dalam bahasa Indonesia untuk customer yang test drive
Mitsubishi XForce Ultimate kemarin tapi belum membuat keputusan pembelian.
Email harus profesional, ramah, dan mendorong tindak lanjut.
"""

initial_draft = generate(initial_prompt, temp=0.7, label="Section 2 — INITIAL DRAFT")


In [ ]:
# --- SELF-CHECK + REVISION ---
self_check_prompt = f"""
Berikut adalah draft email follow-up:

---
{initial_draft}
---

Lakukan self-check dengan langkah berikut:
1. Identifikasi 3 kelemahan email tersebut. Pertimbangkan: apakah terlalu pushy?
   Kurang personal? CTA tidak jelas? Tidak menonjolkan keunggulan XForce
   (misal: desain bold, fitur Mitsubishi Connect, ground clearance tinggi)?
2. Tulis ulang versi yang lebih efektif untuk konversi penjualan,
   mempertahankan tone profesional namun lebih hangat dan persuasif.

Format output:

**Kelemahan yang ditemukan:**
1. ...
2. ...
3. ...

**Revised Email:**
[email yang sudah diperbaiki]
"""

generate(self_check_prompt, temp=0.5, label="Section 2 — SELF-CHECK + REVISION")


### 🔍 Reflection — Section 2

1. Kelemahan mana yang paling tajam ditemukan AI?
2. Apakah revised email terasa lebih natural? Atau malah over-corrected?
3. Coba escalate dengan persona: ubah prompt jadi *"Act as a Mitsubishi Sales Manager dengan 10 tahun pengalaman dan kritik ulang"*. Apakah depth analisisnya berbeda?


---

# Section 3 — Retrieval-Aware

**Skenario:** AI hanya boleh menjawab berdasarkan konteks showroom yang diberikan. Test apakah AI menolak hallucinate untuk pertanyaan yang TIDAK ada di konteks.


In [ ]:
# --- KONTEKS SHOWROOM ---
showroom_context = """
=== KONTEKS SHOWROOM (Maret 2026) ===
Mitsubishi Showroom Jakarta:
- Xpander Ultimate 2024 A/T: 4 unit, harga Rp 295jt
- Xpander Cross Premium 2024: 3 unit, harga Rp 335jt
- XForce Ultimate 2024: 5 unit, harga Rp 405jt
- Pajero Sport Dakar 4x2 2024: 2 unit, harga Rp 720jt

Promo bulan ini:
- DP 10% untuk Xpander
- Gratis service 1 tahun untuk XForce
- Bonus aksesoris Rp 15jt untuk Pajero Sport

Jam operasional: Senin-Sabtu 09:00-18:00.
=== END KONTEKS ===
"""

retrieval_aware_prompt = f"""
{showroom_context}

INSTRUKSI:
Jawab pertanyaan customer HANYA berdasarkan konteks di atas.
Jika informasi tidak ada di konteks, jawab persis:
"Informasi tidak tersedia dalam sumber. Silakan hubungi sales advisor untuk detail lebih lanjut."
Sertakan kutipan kalimat pendukung dari konteks untuk setiap jawaban.

PERTANYAAN CUSTOMER:
1. Berapa unit XForce Ultimate yang tersedia?
2. Promo apa yang berlaku untuk Xpander bulan ini?
3. Apakah showroom buka hari Minggu?
4. Apakah ada promo untuk Mitsubishi Triton bulan ini?

Format jawaban untuk SETIAP pertanyaan:
- Pertanyaan: [pertanyaan]
- Jawaban: [jawaban]
- Sumber: "[kutipan langsung dari konteks ATAU 'Tidak tersedia di konteks']"
"""

generate(retrieval_aware_prompt, temp=0.1, label="Section 3 — RETRIEVAL-AWARE")


### 🔍 Reflection — Section 3

1. Apakah AI menolak halusinasi promo Triton (yang tidak ada di konteks)?
2. Apakah kutipan sumber yang diberikan benar-benar dari konteks atau ada yang dikarang?
3. Apa yang terjadi jika `temp` dinaikkan ke 1.0? Apakah AI tetap patuh?


---

# Section 4 — Final Assignment: Combined Techniques

## 🎯 Objective
Build an optimized prompt for ONE Mitsubishi business use case. Your final prompt **MUST** combine:

| Layer | Pick At Least |
|---|---|
| Reasoning technique | **1 of:** CoT / Self-Check / Retrieval-Aware |
| Engine parameters | **All 3:** Temperature, Top-P, Top-K |
| Cost optimization | **1 of:** Token Budgeting (`max_tokens`) / Prompt Compression |

## 🚗 Choose ONE Use Case
1. **Sales pitch generator** — pitch text untuk customer pada Mitsubishi model tertentu
2. **Lead nurturing email** — follow-up email untuk test drive prospect
3. **Marketplace listing** — deskripsi used Mitsubishi untuk listing online
4. **Customer service response** — balasan untuk komplain delayed service appointment

## 📦 Dummy Data (Mitsubishi)


In [ ]:
DUMMY_DATA = """
=== INVENTORY (Mitsubishi Showroom) ===
- Xpander Ultimate 2024 A/T: 4 unit, KM 8.000, Rp 295jt
- Xpander Cross Premium 2024: 3 unit, KM 5.000, Rp 335jt
- XForce Ultimate 2024: 5 unit, KM 3.500, Rp 405jt
- Pajero Sport Dakar 4x2 2024: 2 unit, KM 6.000, Rp 720jt
- Triton Athlete 4x4 2023: 2 unit, KM 18.000, Rp 525jt

=== CUSTOMER PROFILE ===
- Nama: Budi Santoso
- Usia: 35 tahun
- Keluarga: 4 orang (istri + 2 anak)
- Budget: Rp 280-340jt
- Kebutuhan: MPV/crossover hemat BBM untuk harian + sesekali keluar kota

=== SERVICE CENTER ===
- Avg waktu servis: 2-3 jam
- Ganti oli: Rp 450k
- Tune-up: Rp 950k
- Paket servis berkala: kelipatan KM 10.000
"""

print(DUMMY_DATA)


## ✏️ Your Final Prompt

Edit the cell below — pilih use case, tentukan teknik reasoning, lalu tulis prompt Anda.


In [ ]:
# ============================================================
# TODO: Lengkapi prompt final Anda di bawah
# Pastikan kombinasi berikut ada:
#   - 1 reasoning technique (CoT / Self-Check / Retrieval-Aware)
#   - All 3 engine params (temp, top_p, top_k) — di-set di Section 5
#   - 1 cost optimization (max_tokens = token budgeting,
#     atau compress prompt agar lebih ringkas)
# ============================================================

YOUR_USE_CASE = "Sales pitch generator"   # ← ganti sesuai pilihan Anda
YOUR_REASONING_TECHNIQUE = "CoT"          # ← CoT / Self-Check / Retrieval-Aware
YOUR_COST_STRATEGY = "Token Budgeting"    # ← Token Budgeting / Prompt Compression

YOUR_FINAL_PROMPT = f"""
{DUMMY_DATA}

[TODO: Tulis prompt Anda di sini menggunakan teknik reasoning yang dipilih.

Contoh kerangka untuk CoT pada Sales Pitch:
1. Identifikasi kebutuhan customer (Budi Santoso) dari profile di atas
2. Cocokkan dengan inventory yang sesuai budget & kebutuhan
3. Susun argumen kenapa model tersebut paling cocok
4. Tulis pitch final dalam 3 paragraf]
"""

print("===== USE CASE =====")
print(YOUR_USE_CASE)
print("\n===== REASONING =====")
print(YOUR_REASONING_TECHNIQUE)
print("\n===== COST STRATEGY =====")
print(YOUR_COST_STRATEGY)
print("\n===== FINAL PROMPT =====")
print(YOUR_FINAL_PROMPT)


---

# Section 5 — Run 3 Configurations

Keep the prompt **constant**. Vary only the engine parameters. The reasoning technique stays the same — you're isolating the parameter effect.


In [ ]:
EXPERIMENTS = [
    {"label": "Exp 1 — Safe",     "temp": 0.2, "top_p": 0.5, "top_k": 20,  "max_tokens": 200},
    {"label": "Exp 2 — Balanced", "temp": 0.6, "top_p": 0.8, "top_k": 50,  "max_tokens": 400},
    {"label": "Exp 3 — Creative", "temp": 1.0, "top_p": 1.0, "top_k": 100, "max_tokens": 600},
]

results = {}
for exp in EXPERIMENTS:
    text = generate(
        YOUR_FINAL_PROMPT,
        temp=exp["temp"],
        top_p=exp["top_p"],
        top_k=exp["top_k"],
        max_tokens=exp["max_tokens"],
        label=exp["label"],
    )
    results[exp["label"]] = text

print("\n\n✅ All 3 experiments completed. Hasil tersimpan di dict `results`.")


---

# Section 6 — Analysis Table

Score 1–5 dengan catatan singkat untuk setiap sel.
Edit sel markdown di bawah dan isi tabelnya.

| Criteria | Exp 1 (Safe) | Exp 2 (Balanced) | Exp 3 (Creative) |
|---|---|---|---|
| Clarity |  |  |  |
| Creativity |  |  |  |
| Consistency *(jalankan 2x, bandingkan)* |  |  |  |
| Reasoning Quality *(efektivitas CoT/Self-Check)* |  |  |  |
| Groundedness *(kepatuhan ke konteks, jika Retrieval-Aware)* |  |  |  |
| Token Efficiency *(total tokens dari output)* |  |  |  |
| **Best Output** *(pilih satu)* |  |  |  |


---

# Section 7 — Reflection (200–300 kata)

_(Tulis refleksi Anda di sel markdown ini)_

Pertanyaan panduan:
1. Config mana yang menang untuk use case Anda? Mengapa?
2. Teknik reasoning mana yang paling impactful — CoT, Self-Check, atau Retrieval-Aware?
3. Apa yang mengejutkan dari hasil eksperimen?
4. Bagaimana cost optimization (max_tokens / compression) berdampak pada kualitas output?
5. Untuk production deployment di Mitsubishi, config mana yang akan Anda rekomendasikan? Apa risikonya?

**[Tulis refleksi 200-300 kata di sini]**


---

# ✅ Submission Checklist

Sebelum submit ke Google Classroom, pastikan:

- [ ] Setup berhasil — API key dari Colab Secrets, **TIDAK** hardcoded
- [ ] **Section 1 (CoT)** — kedua versi (without/with) sudah dijalankan
- [ ] **Section 2 (Self-Check)** — initial draft + revised version sudah dijalankan
- [ ] **Section 3 (Retrieval-Aware)** — keempat pertanyaan dijawab dengan citation
- [ ] **Section 4** — `YOUR_USE_CASE`, `YOUR_REASONING_TECHNIQUE`, `YOUR_COST_STRATEGY`, dan `YOUR_FINAL_PROMPT` sudah diisi
- [ ] **Section 5** — 3 eksperimen sudah dijalankan
- [ ] **Section 6** — analysis table terisi lengkap (tidak ada sel kosong)
- [ ] **Section 7** — refleksi 200–300 kata sudah ditulis
- [ ] File rename: `NamaLengkap_Sesi25_Assignment.ipynb`
- [ ] Upload ke Google Classroom sebelum deadline

**Selamat mengerjakan! 🚗💨**
